<a href="https://colab.research.google.com/github/Shivanaga7sheelavantar/AI-Website-Chatbot/blob/main/game_development_crew_initial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
!pip install -U crewai

In [46]:
!pip install "crewai[google-genai]"

In [47]:
!pip install "crewai[tools]"

In [48]:
from crewai import Agent,LLM, Task,Process, Crew

In [49]:
from google.colab import userdata

In [50]:
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=GEMINI_API_KEY
)

TimeoutException: Requesting secret GEMINI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

#**Creative Game Designer Agent**
```
   role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
   
   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
```

In [ ]:
game_designer = Agent(
   role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",

   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
     verbose=True,
     llm=llm,
   )

In [ ]:
from crewai_tools import SerperDevTool
serper_api_key = userdata.get('SERPER_APIKEY')
search_tool = SerperDevTool(api_key = serper_api_key)

#**Senior Python Game Developer Agent**

```
role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
```

In [ ]:
senior_engineer = Agent(role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
       verbose=True,
       llm=llm,
       tools=[search_tool]
)

#**QA Engineer & Code Reviewer Agent**

    role="QA Engineer & Code Reviewer",

    goal="Test, review, and improve the code for bugs, playability, and completeness",
    
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",


In [ ]:
qa_engineer =Agent(role="QA Engineer & Code Reviewer",

goal="Test, review, and improve the code for bugs, playability, and completeness",

backstory=
    "You are a meticulous QA engineer and code reviewer."
    "You carefully check:"
    "- Does the code run without errors?"
    "- Does it implement ALL the designed features?"
    "- Is it fun/playable? Any obvious balance issues?"
    "- Code style, variable names, comments"
    "Suggest fixes or small improvements and output the FINAL improved code.",
    verbose=True,
    llm=llm
    )

### **Game Designing Task**

    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",

In [ ]:
task_design = Task(
   description=
    "Take the user's game idea: {game_idea}"
    "1. Clarify and expand it into a fun, simple 2D game"
    "2. Describe: objective, controls, entities, win/lose"
    "3. Keep scope small (one level, basic mechanics)"
    "Output format:"
    "## Game Design Document"
    "- Title: ..."
    "- Genre: ..."
    "- Objective: ..."
    "- Controls: ..."
    "- Entities: ..."
    "- Mechanics: ...",
   expected_output="A clear markdown Game design Document",
   agent=game_designer
)

### **Coding Task**

    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main",
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",

In [ ]:
task_code = Task(
    description=
    "Using the game design from the previous task\n"
    "Write a COMPLETE, standalone Python script using Pygame that implements the game.\n"
    "- Include import pygame, sys, random (if needed)\n"
    "- Full game loop, init, events, update, draw\n"
    "- Make it runnable with python game.py\n"
    "- Add simple comments\n"
    "- The main game loop must be exposed in the python code, it should not be inside any function like main\n"
    "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",

    expected_output="A complete runnable pygame python srcipt",
    agent=senior_engineer,
    context=[task_design]
)

###**Review Task**

    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    

In [ ]:
task_review = Task(description=
    "Review the Python code from the previous task."
    "1. Check for syntax/runtime errors"
    "2. Verify it matches the design document"
    "3. Test mentally: does it have init, loop, quit handling, drawing?"
    "4. Suggest fixes/improvements if needed"
    "5. Output the FINAL, improved, ready-to-run code"
    "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final Polished, runnable pygame python script and instructions on how to play the game",
    agent=qa_engineer,
    context=[task_design,task_code]
    )

In [ ]:
game_crew=Crew(
    agents=[game_designer,senior_engineer,qa_engineer],
    tasks=[task_design,task_code,task_review],
    process=Process.sequential,
    verbose=True
)

In [ ]:
game_idea = "A fun endless runner where a character jumps over obstacles"
result = game_crew.kickoff(inputs={"game_idea": game_idea})
print(result)

In [ ]:
%%writefile main.py
import pygame
import sys
import random

# Initialize Pygame
pygame.init()

# --- Game Constants ---
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 400
GROUND_HEIGHT = 50  # Height of the ground rectangle

PLAYER_SIZE = 30

OBSTACLE_WIDTH_SMALL = 20
OBSTACLE_HEIGHT_SMALL = 30
OBSTACLE_WIDTH_TALL = 20
OBSTACLE_HEIGHT_TALL = 60

# Minimum and maximum horizontal distance (in pixels scrolled) between consecutive obstacles
OBSTACLE_MIN_DISTANCE = 200
OBSTACLE_MAX_DISTANCE = 400

GRAVITY = 0.8         # Strength of gravity pulling the player down
JUMP_STRENGTH = -15   # Initial upward velocity when player jumps

INITIAL_GAME_SPEED = 5                     # Initial speed at which the world scrolls
SPEED_INCREASE_INTERVAL_MS = 1000          # How often (in ms) the game speed increases
SPEED_INCREASE_AMOUNT = 0.2                # How much the game speed increases each interval

# --- Colors ---
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (0, 0, 255)
GREEN = (0, 255, 0)
RED = (255, 0, 0)

# --- Set up the display ---
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Pixel Hop")

# --- Fonts ---
font = pygame.font.Font(None, 36)
game_over_font = pygame.font.Font(None, 72)
restart_font = pygame.font.Font(None, 24)

# --- Player Class ---
class Player:
    """Represents the player character, handling movement, jumping, and drawing."""
    def __init__(self):
        # Player's rectangle for position and collision. X position is fixed.
        self.rect = pygame.Rect(50, SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_SIZE, PLAYER_SIZE, PLAYER_SIZE)
        self.y_velocity = 0
        self.is_jumping = False
        self.on_ground = True

    def jump(self):
        """Initiates a jump if the player is currently on the ground."""
        if self.on_ground:
            self.y_velocity = JUMP_STRENGTH
            self.is_jumping = True
            self.on_ground = False

    def update(self):
        """Updates player's vertical position based on gravity and velocity."""
        # Apply gravity to vertical velocity
        self.y_velocity += GRAVITY
        # Update player's vertical position
        self.rect.y += self.y_velocity

        # Check for ground collision
        if self.rect.bottom >= SCREEN_HEIGHT - GROUND_HEIGHT:
            self.rect.bottom = SCREEN_HEIGHT - GROUND_HEIGHT # Snap player to ground level
            self.y_velocity = 0                              # Stop vertical movement
            self.is_jumping = False
            self.on_ground = True

    def draw(self, screen):
        """Draws the player character on the screen."""
        pygame.draw.rect(screen, BLUE, self.rect)

# --- Obstacle Class ---
class Obstacle:
    """Represents an obstacle, handling its type, movement, and drawing."""
    def __init__(self, x_pos, type_is_tall):
        # Determine obstacle dimensions based on type
        if type_is_tall:
            self.width = OBSTACLE_WIDTH_TALL
            self.height = OBSTACLE_HEIGHT_TALL
        else:
            self.width = OBSTACLE_WIDTH_SMALL
            self.height = OBSTACLE_HEIGHT_SMALL

        # Position the obstacle just above the ground, spawning off-screen to the right
        self.rect = pygame.Rect(x_pos, SCREEN_HEIGHT - GROUND_HEIGHT - self.height, self.width, self.height)

    def update(self, speed):
        """Moves the obstacle from right to left based on game speed."""
        self.rect.x -= speed

    def draw(self, screen):
        """Draws the obstacle on the screen."""
        pygame.draw.rect(screen, RED, self.rect)

# --- Game State Reset Function ---
def reset_game_state():
    """Resets all game variables to their initial state for a new game."""
    global player, obstacles, game_speed, score_start_time, game_over, \
           current_scrolled_distance_since_last_obstacle, next_obstacle_distance_threshold, last_speed_increase_time

    player = Player()
    obstacles = []
    game_speed = INITIAL_GAME_SPEED
    score_start_time = pygame.time.get_ticks() # Reset the timer for score calculation
    game_over = False

    # Reset variables for obstacle generation
    current_scrolled_distance_since_last_obstacle = 0
    next_obstacle_distance_threshold = random.randint(OBSTACLE_MIN_DISTANCE, OBSTACLE_MAX_DISTANCE)

    last_speed_increase_time = pygame.time.get_ticks()

# --- Initial Game State ---
# Initialize game variables for the first play
player = Player()
obstacles = []
game_speed = INITIAL_GAME_SPEED
score_start_time = pygame.time.get_ticks() # Timestamp when the current game started for scoring
game_over = False

# Variables to manage obstacle generation based on scrolled distance
# current_scrolled_distance_since_last_obstacle: accumulates the distance the world has scrolled
#   since the last obstacle was spawned.
# next_obstacle_distance_threshold: the random distance the world must scroll before the next obstacle spawns.
current_scrolled_distance_since_last_obstacle = 0
next_obstacle_distance_threshold = random.randint(OBSTACLE_MIN_DISTANCE, OBSTACLE_MAX_DISTANCE)

last_speed_increase_time = pygame.time.get_ticks() # Timestamp for last game speed increase

# --- Game Loop ---
clock = pygame.time.Clock()
running = True

while running:
    # --- Event Handling ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False # Exit the game loop if the user closes the window
        if event.type == pygame.KEYDOWN:
            if not game_over:
                if event.key == pygame.K_SPACE:
                    player.jump() # Player jumps if game is active and space is pressed
            else: # Game is over, check for restart input
                if event.key == pygame.K_SPACE:
                    reset_game_state() # Restart the game

    # --- Game State Update (only if game is not over) ---
    if not game_over:
        player.update()

        # Update obstacles' positions and remove ones that have moved completely off-screen
        # Iterate over a copy of the list to safely remove items during iteration
        for obstacle in list(obstacles):
            obstacle.update(game_speed)
            if obstacle.rect.right < 0: # Obstacle has moved completely off-screen to the left
                obstacles.remove(obstacle)

        # Update obstacle spawn progress and generate new obstacles
        current_scrolled_distance_since_last_obstacle += game_speed # Accumulate scrolled distance
        if current_scrolled_distance_since_last_obstacle >= next_obstacle_distance_threshold:
            is_tall = random.choice([True, False]) # Randomly choose obstacle type (small or tall)
            new_obstacle = Obstacle(SCREEN_WIDTH, is_tall) # Spawn new obstacle off-screen to the right
            obstacles.append(new_obstacle)

            # Reset progress for the next gap and determine the new random spawn threshold
            current_scrolled_distance_since_last_obstacle = 0
            next_obstacle_distance_threshold = random.randint(OBSTACLE_MIN_DISTANCE, OBSTACLE_MAX_DISTANCE)

        # Update score based on elapsed time (1 point per second)
        current_time = pygame.time.get_ticks()
        score = (current_time - score_start_time) // 1000

        # Increase game speed gradually over time
        if current_time - last_speed_increase_time >= SPEED_INCREASE_INTERVAL_MS:
            game_speed += SPEED_INCREASE_AMOUNT
            last_speed_increase_time = current_time

        # Collision detection: Check if player collides with any obstacle
        for obstacle in obstacles:
            if player.rect.colliderect(obstacle.rect):
                game_over = True # End the game on collision

    # --- Drawing ---
    screen.fill(WHITE) # Clear the screen with a white background

    # Draw ground (a continuous green rectangle at the bottom)
    pygame.draw.rect(screen, GREEN, (0, SCREEN_HEIGHT - GROUND_HEIGHT, SCREEN_WIDTH, GROUND_HEIGHT))

    # Draw player character
    player.draw(screen)

    # Draw all active obstacles
    for obstacle in obstacles:
        obstacle.draw(screen)

    # Draw current score display
    score_text = font.render(f"Score: {score}", True, BLACK)
    screen.blit(score_text, (SCREEN_WIDTH - score_text.get_width() - 10, 10))

    # Display Game Over screen if game_over is True
    if game_over:
        game_over_text = game_over_font.render("GAME OVER", True, BLACK)
        final_score_text = font.render(f"Final Score: {score}", True, BLACK)
        restart_text = restart_font.render("Press SPACE to Restart", True, BLACK)

        # Center the game over messages on the screen
        screen.blit(game_over_text, (SCREEN_WIDTH // 2 - game_over_text.get_width() // 2, SCREEN_HEIGHT // 2 - 50))
        screen.blit(final_score_text, (SCREEN_WIDTH // 2 - final_score_text.get_width() // 2, SCREEN_HEIGHT // 2 + 20))
        screen.blit(restart_text, (SCREEN_WIDTH // 2 - restart_text.get_width() // 2, SCREEN_HEIGHT // 2 + 60))

    pygame.display.flip() # Update the entire screen to show what has been drawn

    # Cap the frame rate to 60 frames per second to ensure consistent game speed across machines
    clock.tick(60)

# --- Clean up Pygame ---
pygame.quit()
sys.exit()

# **Do not edit any of the below code**

In [ ]:
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')

In [ ]:
!pip install pygbag pyngrok -q

# **How to get pyngrok API Key?**
- Go to https://dashboard.ngrok.com/get-started/your-authtoken
- Copy your authtoken
- Run this cell with YOUR token:

In [ ]:
!ngrok authtoken '3AkWizLZX4dkBco0jFoO0GqwAnM_7eRwRHgQUGWF7FgZWqRbT'

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', 'main.py'],
    capture_output=True, text=True
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Start HTTP server
    print("\n🚀 Starting server...")
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000', '--directory', '/content/build/web'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Step 3: Create ngrok tunnel
    try:
        print("🌐 Creating public URL...")
        public_url = ngrok.connect(8000)

        print("\n" + "="*60)
        print("🎮 YOUR GAME IS READY!")
        print("="*60)
        print(f"\n🔗 Click here to play: {public_url}")
        print("\n📝 How to play:")
        print("   • Press SPACE or Click to jump")
        print("="*60)

    except Exception as e:
        print(f"\n❌ Error creating tunnel: {e}")
        print("\n💡 Make sure you ran Cell 3 with a valid ngrok token")